<a href="https://colab.research.google.com/github/Nithya-collab/Qiskit-Quantum-/blob/main/Classic%20A*%20VS%20QAOA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Small Graph shotest path finding using classical A Star**

In [ ]:
import numpy as np

# 2x2 grid
# 2 = Start, 3 = Goal
grid = np.array([
    [2, 0],
    [0, 3]
])

start = (0, 0)
goal = (1, 1)
moves = [(-1,0),(1,0),(0,-1),(0,1)]

**Classical A star**

In [ ]:
import heapq

def heuristic(a, b):
    return abs(a[0]-b[0]) + abs(a[1]-b[1])

def astar(grid, start, goal):
    rows, cols = grid.shape
    open_list = []
    heapq.heappush(open_list,(0,start))

    came_from = {}
    cost_so_far = {start:0}

    while open_list:
        current = heapq.heappop(open_list)[1]

        if current == goal:
            break

        for move in moves:
            next_node = (current[0]+move[0], current[1]+move[1])

            if 0 <= next_node[0] < rows and 0 <= next_node[1] < cols:
                new_cost = cost_so_far[current] + 1

                if next_node not in cost_so_far or new_cost < cost_so_far[next_node]:
                    cost_so_far[next_node] = new_cost
                    priority = new_cost + heuristic(goal,next_node)

                    heapq.heappush(open_list,(priority,next_node))
                    came_from[next_node] = current

    # reconstruct path
    path = []
    node = goal
    while node != start:
        path.append(node)
        node = came_from[node]
    path.append(start)
    path.reverse()

    return path

# Run A*
import time
start_time = time.time()
path = astar(grid, start, goal)
end_time = time.time()

astar_time = end_time - start_time
astar_length = len(path)

print("A* Path:", path)
print("A* Length:", astar_length)
print("A* Time:", astar_time)

A* Path: [(0, 0), (0, 1), (1, 1)]
A* Length: 3
A* Time: 9.822845458984375e-05


**Cost Hamiltonian**

In [ ]:
!pip install qiskit qiskit-aer qiskit-algorithms

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.6 MB/s eta 0:00:00


In [1]:
from qiskit.quantum_info import SparsePauliOp
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import StatevectorSampler

# 4 qubits (for 4 nodes)

cost_operator = SparsePauliOp.from_list([
    # minimize number of nodes (path length)
    ("ZIII", 1.0),
    ("IZII", 1.0),
    ("IIZI", 1.0),
    ("IIIZ", 1.0),

    # enforce start (node 0)
    ("ZIII", -2.0),

    # enforce goal (node 3)
    ("IIIZ", -2.0)
])

ModuleNotFoundError: No module named 'qiskit'

**QAOA - (Quantum Approximate Optimization Algo)**

In [ ]:
sampler = StatevectorSampler()
optimizer = COBYLA(maxiter=100)

qaoa = QAOA(
    sampler=sampler,
    optimizer=optimizer,
    reps=1
)

result = qaoa.compute_minimum_eigenvalue(cost_operator)

print("QAOA Energy:", result.eigenvalue)
print("QAOA State:", result.eigenstate)

/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)


QAOA Energy: -4.0
QAOA State: {'0110': 1.0}


Decode QAOA Result

In [ ]:
import numpy as np

# Get most probable bitstring
state = result.eigenstate
probabilities = state

best = max(probabilities, key=probabilities.get)

print("Best bitstring:", best)

Best bitstring: 0110


**Comparison Table**

In [ ]:
import pandas as pd

data = {
    "Method": ["A*", "QAOA"],
    "Path Length": [astar_length, "from_bitstring"],
    "Execution Time": [astar_time, "qaoa_time"]
}

df = pd.DataFrame(data)
print(df)

  Method     Path Length Execution Time
0     A*               3       0.000098
1   QAOA  from_bitstring      qaoa_time
